# Cassini CDA Dust Analyzer: Ablation Study (Batch Mode)

This notebook splits labeled data into a fixed test dataset (up to 100 spectra per class, stratified) and a flexible pool for few-shot training. 

To scale the evaluation efficiently and avoid online API rate limits, it consolidates all few-shot levels ($k \in [1, 2, 4, 8, 12, 16, 24, 32]$) into a **single Vertex AI Batch Prediction job** (up to 5,600 test classification instances). 

Once the batch job completes, the predictions are parsed and saved to a **single consolidated Parquet file** containing `sclk`, `true_label`, `predicted_label`, and `k`.

In [5]:
import os
import sys
import asyncio
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from google import genai
import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger("google_genai").setLevel(logging.WARNING)

# Ensure we can import from the workspace
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import importlib
import scripts.run_ablation_study
importlib.reload(scripts.run_ablation_study)

from scripts.run_ablation_study import (
    preprocess_huggingface_data,
    generate_nested_explanations_pool,
    create_ablation_batch_input_file,
    submit_and_poll_batch_job,
    download_and_parse_batch_results,
    plot_and_save_ablation_results
)

from cda_dust_agent.config import Config

# Load environment variables and configurations
load_dotenv(dotenv_path="../.env")
configs = Config()
client = genai.Client()
model_id = configs.agent_settings.model

print(f"Using Model: {model_id}")

Using Model: gemini-3.1-pro-preview


In [6]:
# 1. Fetch and Preprocess Data
print("Loading data...")
full_df = preprocess_huggingface_data("H")
print(f"Total available rows: {len(full_df)}")

# Display the class distribution to check preprocessing
print("\nClass counts:")
print(full_df['class'].value_counts())

Loading data...
Loading processed H dataset(s) generated by DataFetchAndParseAgent...
Total available rows: 6986

Class counts:
class
Noise    4442
1        1183
2         829
3         246
4         210
5          42
5-Na       34
Name: count, dtype: int64


In [7]:
# 2. Split Data: Fixed Test Set and Remaining Pool
TEST_SIZE_PER_CLASS = 100

test_dfs = []
pool_dfs = []

# Only keep classes that have at least (TEST_SIZE_PER_CLASS + 1) examples
for cls, group in full_df.groupby('class'):
    if len(group) >= TEST_SIZE_PER_CLASS + 1:
        # Shuffle
        group = group.sample(frac=1, random_state=42).reset_index(drop=True)
        test_dfs.append(group.head(TEST_SIZE_PER_CLASS))
        pool_dfs.append(group.tail(len(group) - TEST_SIZE_PER_CLASS))
    else:
        print(f"Skipping class '{cls}' - not enough samples ({len(group)})")

test_df = pd.concat(test_dfs, ignore_index=True)
pool_df = pd.concat(pool_dfs, ignore_index=True)

print(f"\nFinal Test Set Size: {len(test_df)} (up to {TEST_SIZE_PER_CLASS} per class)")
print(f"Total remaining pool for training: {len(pool_df)}")

Skipping class '5' - not enough samples (42)
Skipping class '5-Na' - not enough samples (34)

Final Test Set Size: 500 (up to 100 per class)
Total remaining pool for training: 6410


In [ ]:
# 3. Pre-generate Nested VLM Explanations Pool (Up to 32 per class)
# Since the total number of explanations needed is small (<= 224), we generate them online concurrently.
explanations_pool = await generate_nested_explanations_pool(client, model_id, pool_df, max_k=32)

Sampling up to 32 examples per class from training pool...
Generating 160 reference explanations online...
Successfully generated 160 explanations.


: 

In [ ]:
# 4. Generate Consolidated Batch Requests JSONL File
k_values = [1, 2, 4, 8, 12, 16, 24, 32]
local_jsonl_path = "../cda_dust_agent/data/input/ablation_requests.jsonl"

create_ablation_batch_input_file(test_df, explanations_pool, k_values, local_jsonl_path)

Generating consolidated batch request file for k=[1, 2, 4, 8, 12, 16, 24, 32]...
  -> Building request payloads for k=1 shots...
  -> Building request payloads for k=2 shots...
  -> Building request payloads for k=4 shots...
  -> Building request payloads for k=8 shots...
  -> Building request payloads for k=12 shots...
  -> Building request payloads for k=16 shots...
  -> Building request payloads for k=24 shots...


In [ ]:
# 5. Submit Batch Job to Vertex AI and Poll for Completion
# Note: Vertex AI Batch prediction jobs take 5-10 minutes to spin up.
local_jsonl_path = "../cda_dust_agent/data/input/ablation_requests.jsonl"
job_status_data = submit_and_poll_batch_job(configs, local_jsonl_path)

In [ ]:
# 6. Download and Parse Batch Results into a Consolidated Parquet File
output_parquet_path = "../cda_dust_agent/data/results/all_ablation_results.parquet"
results_df = download_and_parse_batch_results(configs, job_status_data, test_df, output_parquet_path)

# Show a preview of the consolidated results
results_df.head(10)

In [ ]:
# 7. Compute Metrics, Generate Accuracy Curve & Confusion Matrices
plot_and_save_ablation_results(results_df, "ablation_results")